# E-Commerce Customer Churn Prediction
## Project Summary

This project builds an end-to-end machine learning system to predict customer churn 
using real transaction data from Olist, a Brazilian e-commerce marketplace. The 
business problem, methodology, findings, and model decisions are documented here 
as a single reference.

The core question we're trying to answer:

> Given a customer's first purchase history, can we predict whether they will 
> ever buy again?

## The Dataset

The data comes from Olist, a real Brazilian e-commerce platform, and contains 100,000 orders placed between September 2016 and October 2018. It was downloaded from Kaggle and is freely available.

The dataset is split across 9 tables — orders, customers, order items, payments, reviews, products, sellers, geolocation, and category translations. Each table connects to others through shared IDs, meaning the first challenge was understanding how to join them meaningfully rather than just throwing everything together.

Only 6 of the 9 tables were used. Seller location and geolocation data were excluded as they don't directly influence individual customer behavior, and the category translation table was excluded as it only provides English labels for Portuguese category names which don't affect modeling.

## Defining Churn

This was the most important decision in the entire project.

93,358 unique customers placed orders, and 90,557 of them — 97% — bought exactly once and never returned. Because almost nobody comes back, traditional churn modeling doesn't apply cleanly here. You can't define churn as "stopped buying after being active" when most customers were never active in the first place.

Instead churn was defined as: a customer who did not make a second purchase within 6 months of their first order.

Customers whose first order was within the last 6 months of the dataset were excluded from labeling — they hadn't been given enough time to potentially return, so labeling them as churned would be misleading.

This left 55,364 customers available for modeling, with a churn rate of 96%.

## Feature Engineering

21 features were built from the raw tables, grouped into four categories:

**RFM features** capture how recently, how often, and how much each customer spent. Recency measures days since last purchase. Frequency measures total orders. Monetary measures total spend across all orders.

**Delivery features** capture the customer's delivery experience. Average delivery delay and maximum delivery delay measure how many days late (positive) or early (negative) orders arrived relative to the estimated date. Late deliveries counts how many orders arrived after the promised date.

**Payment features** capture payment behavior. Average installments measures how many installments customers typically used. Unique payment types counts how many different payment methods were used. Used voucher flags whether the customer ever paid with a voucher.

**Product features** capture what customers bought. Average and maximum item price measure spend per item. Average product weight and freight value capture the physical size of purchases. 

**Review features** capture satisfaction signals. Average review score and minimum review score measure overall and worst-case satisfaction. Gave bad review flags customers who ever left a score of 2 or below.

Three features were removed after discovering data leakage — frequency, unique categories, and total items. These features directly encoded the churn label because a customer who bought once by definition has frequency of 1, fewer categories, and fewer items. Keeping them would have given the model the answer rather than teaching it to find patterns.

## Modeling

Three models were trained and compared:

**Logistic Regression** was used as the baseline. It achieved an AUC-ROC of 0.9956 using balanced class weights to handle the 96% churn imbalance. Features were scaled with StandardScaler before training.

**LightGBM** was the main model. Out of the box it improved AUC to 0.9987. LightGBM was chosen over XGBoost because it is faster on this dataset size, handles categorical-like features better, and is widely used in industry for tabular data problems.

**Tuned LightGBM with Optuna** was the final model. Optuna ran 50 trials of Bayesian hyperparameter optimization and found the best combination of learning rate, tree depth, number of leaves, and subsampling parameters. The final AUC reached 0.9991.

Class imbalance was handled using class weights rather than oversampling. Oversampling techniques like SMOTE create synthetic samples which can introduce artificial patterns into the data. Class weights instead penalize the model more heavily for misclassifying the minority class, which is a cleaner and more honest approach.

The best hyperparameters from Optuna were: 633 estimators, learning rate of 0.052, 39 leaves, max depth of 9, minimum child samples of 64, subsample of 0.63, and column sample of 0.92.

## Key Findings

**Delivery delay is the single biggest driver of churn.** Both maximum and average delivery delay ranked at the top of the SHAP feature importance chart, far above everything else. Customers who received orders late were strongly predicted to churn. Customers who received orders early were strongly predicted to return. This finding has a direct business implication — investing in logistics reliability is more valuable for customer retention than discounts or marketing campaigns.

**High spenders are not necessarily loyal customers.** Monetary value showed significant overlap between churned and retained customers in the distribution analysis. A customer who spent 500 BRL once is not more likely to return than someone who spent 100 BRL. Spend alone is not a reliable loyalty signal.

**Heavy product purchases are a strong churn signal.** The individual SHAP explanation for a churned customer revealed that buying a very heavy, expensive item — likely a large one-off purchase like furniture or an appliance — was the dominant factor pushing toward churn. These customers are making a one-time purchase and are unlikely to return regardless of retention efforts.

**Good delivery experience is the strongest loyalty signal.** The retained customer explanation showed that receiving orders 15 days early on average was the single biggest factor pushing toward retention. Under-promising and over-delivering on delivery dates appears to be one of the most effective ways to build customer loyalty.

**Review scores matter but are downstream of delivery.** Average review score ranked 5th in feature importance. This suggests that bad reviews are often a consequence of poor delivery rather than an independent driver of churn — fix the delivery, and the reviews improve too.

## What I Would Do Differently

**Use more recent data.** The Olist dataset covers 2016 to 2018. Consumer behavior in e-commerce has changed significantly since then, particularly in SEA markets where mobile commerce and same-day delivery have become standard expectations.

**Add seller-level features.** Seller quality likely influences whether a customer returns. A customer who had a bad experience with a specific seller might still return to the platform to buy from a different seller. Including seller ratings and fulfillment rates could improve the model.

**Add geolocation features.** Distance between customer and seller likely affects delivery time and satisfaction. Customers in remote areas may have systematically worse delivery experiences which the current model attributes to delay without understanding the underlying cause.

**Try a survival analysis approach.** Instead of a binary churn label, survival analysis models the time until a customer churns. This would give a richer output — not just "will they churn" but "when are they most likely to churn" — which is more actionable for retention campaign timing.

**Build a proper retention intervention model.** Predicting churn is only half the problem. The other half is knowing which customers will respond to a retention campaign. A two-model system — one for churn probability, one for treatment response — would be more useful in practice.

## Tools Used

- **Python** — pandas, numpy for data manipulation
- **LightGBM** — main model
- **Optuna** — hyperparameter tuning
- **SHAP** — model explainability
- **Matplotlib, Seaborn** — visualization
- **Streamlit** — interactive dashboard
- **Docker** — containerization
- **GitHub** — version control and portfolio hosting